# 🏀 NBA Players Deep EDA & Data Cleaning

This notebook provides comprehensive exploratory data analysis and cleaning of the NBA players raw dataset. It systematically examines data quality, identifies issues, and implements cleaning strategies to prepare the data for analytics.

## Table of Contents
1. **Import Libraries & Load Data**
2. **Basic Data Overview & Structure**
3. **Column-by-Column Deep Dive**
4. **Missing Values Analysis**
5. **Data Type Validation**
6. **Duplicate Records Detection**
7. **Outlier Detection & Analysis**
8. **Data Quality Issues**
9. **Text Data Cleaning**
10. **Categorical Data Analysis**
11. **Numerical Data Distribution**
12. **Cross-Column Validation**
13. **Data Cleaning Implementation**
14. **Cleaned Data Validation & Export**

---

**Dataset**: 2024 NBA Players  
**Objective**: Clean and prepare players data for Pacers analytics pipeline  
**Tech Stack**: pandas, numpy, matplotlib, seaborn, scipy

## 1. Import Required Libraries and Load Data

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [ ]:
# Navigate to project root if needed
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    os.chdir(current_dir.parent)
    print(f"📁 Changed directory to: {Path.cwd()}")

# Load the NBA players dataset
data_file = Path('data/raw/2024_NBA_Players.parquet')

if data_file.exists():
    players_df = pd.read_parquet(data_file)
    print(f"✅ Successfully loaded NBA Players dataset")
    print(f"📄 File: {data_file}")
    print(f"💾 File size: {data_file.stat().st_size / (1024*1024):.2f} MB")
    print(f"📊 Dataset shape: {players_df.shape}")
else:
    print(f"❌ File not found: {data_file}")
    print("Please ensure the data file exists in the correct location.")

# Create a backup copy for comparison later
players_original = players_df.copy()
print(f"💾 Created backup copy: players_original")

## 2. Basic Data Overview and Structure Analysis

In [ ]:
# Dataset Overview
print("🏀 NBA PLAYERS DATASET OVERVIEW")
print("=" * 50)
print(f"📊 Shape: {players_df.shape[0]:,} rows × {players_df.shape[1]} columns")
print(f"💾 Memory usage: {players_df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
print(f"🏷️ Index type: {type(players_df.index)}")
print(f"📋 Column names: {len(players_df.columns)} total")

print("\n📋 COLUMN OVERVIEW")
print("-" * 30)
for i, col in enumerate(players_df.columns, 1):
    print(f"{i:2d}. {col} , {players_df[col].dtype}, Nulls: {players_df[col].isnull().sum()}, Unique: {players_df[col].nunique()}")


In [ ]:
# Data Types and Basic Info
print("🔍 DATA TYPES ANALYSIS")
print("=" * 30)

# Create comprehensive data type summary
dtype_summary = pd.DataFrame({
    'Column': players_df.columns,
    'Data_Type': players_df.dtypes.astype(str),
    'Non_Null_Count': players_df.count(),
    'Null_Count': players_df.isnull().sum(),
    'Null_Percentage': round((players_df.isnull().sum() / len(players_df)) * 100, 2),
    'Memory_Usage_KB': round(players_df.memory_usage(deep=True)[1:] / 1024, 2)
})

# Add unique value counts
dtype_summary['Unique_Values'] = [players_df[col].nunique() for col in players_df.columns]
dtype_summary['Sample_Values'] = [str(players_df[col].dropna().iloc[:3].tolist())[:50] + "..." 
                                  if len(players_df[col].dropna()) > 0 else "No data" 
                                  for col in players_df.columns]

display(dtype_summary)

# Memory usage breakdown by data type
print(f"\n💾 MEMORY USAGE BY DATA TYPE")
print("-" * 35)
memory_by_type = players_df.select_dtypes(include=[np.object_]).memory_usage(deep=True).sum() / 1024
print(f"Object columns: {memory_by_type:.2f} KB")

for dtype in ['int64', 'float64', 'bool']:
    if len(players_df.select_dtypes(include=[dtype]).columns) > 0:
        memory = players_df.select_dtypes(include=[dtype]).memory_usage(deep=True).sum() / 1024
        print(f"{dtype} columns: {memory:.2f} KB")

## 3. Column-by-Column Deep Dive

In [ ]:
def analyze_column(df, column_name, show_plots=True):
    """
    Comprehensive analysis of a single column
    """
    print(f"🔍 ANALYZING COLUMN: {column_name}")
    print("=" * 50)
    
    col_data = df[column_name]
    
    # Basic statistics
    print(f"Data Type: {col_data.dtype}")
    print(f"Non-null values: {col_data.count():,} ({col_data.count()/len(df)*100:.1f}%)")
    print(f"Null values: {col_data.isnull().sum():,} ({col_data.isnull().sum()/len(df)*100:.1f}%)")
    print(f"Unique values: {col_data.nunique():,}")
    print(f"Memory usage: {col_data.memory_usage(deep=True)/1024:.2f} KB")
    
    if col_data.nunique() > 0:
        # Value counts for categorical or low-cardinality columns
        if col_data.nunique() <= 20:
            print(f"\n📊 VALUE COUNTS:")
            print("-" * 20)
            value_counts = col_data.value_counts(dropna=False)
            for val, count in value_counts.head(10).items():
                percentage = (count / len(df)) * 100
                print(f"{str(val)[:30]:30} {count:6,} ({percentage:5.1f}%)")
            
            if len(value_counts) > 10:
                print(f"... and {len(value_counts) - 10} more values")
        
        # Statistical summary for numerical columns
        if pd.api.types.is_numeric_dtype(col_data):
            print(f"\n📈 STATISTICAL SUMMARY:")
            print("-" * 25)
            desc = col_data.describe()
            for stat, value in desc.items():
                print(f"{stat:10}: {value:12.2f}")
                
            # Additional statistics
            print(f"{'Skewness':10}: {col_data.skew():12.2f}")
            print(f"{'Kurtosis':10}: {col_data.kurtosis():12.2f}")
        
        # Sample values
        print(f"\n🔍 SAMPLE VALUES:")
        print("-" * 18)
        sample_vals = col_data.dropna().head(5).tolist()
        for i, val in enumerate(sample_vals, 1):
            print(f"{i}. {str(val)[:50]}")
    
    print("\n" + "="*50 + "\n")

# Analyze first few columns as examples
for col in players_df.columns[:]:
    analyze_column(players_df, col, show_plots=False)